# 예제 03. 층 고정과 미세조정
빅데이터프로그래밍 · 10주차

## 목표
- `requires_grad` 로 층을 고정한다
- 세 가지 방식의 학습 대상 파라미터 수를 비교한다
- 언제 어느 방식을 쓸지 안다

| 방식 | 학습하는 부분 | 언제 |
| --- | --- | --- |
| 특징 추출만 사용 | 마지막 계층만 | 데이터가 아주 적을 때 |
| 일부 계층만 학습 | 뒤쪽 몇 층 + 마지막 | 중간 |
| 전체 미세조정 | 전부 | 데이터가 충분할 때 |


In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import pandas as pd

def count(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train


## 1. 방식 A — 특징 추출만 사용 (전부 고정, fc만 학습)


In [ ]:
a = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for p in a.parameters():
    p.requires_grad = False          # 전부 고정

a.fc = nn.Linear(a.fc.in_features, 5)  # 새 계층은 자동으로 requires_grad=True

t, tr = count(a)
print(f"전체 {t:,}개 중 학습 {tr:,}개 ({tr/t*100:.2f}%)")


## 2. 방식 B — 뒤쪽 일부만 학습


In [ ]:
b = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for p in b.parameters():
    p.requires_grad = False

for p in b.layer4.parameters():       # layer4 만 풀어줍니다
    p.requires_grad = True

b.fc = nn.Linear(b.fc.in_features, 5)

t, tr = count(b)
print(f"전체 {t:,}개 중 학습 {tr:,}개 ({tr/t*100:.2f}%)")


## 3. 방식 C — 전체 미세조정


In [ ]:
c = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
c.fc = nn.Linear(c.fc.in_features, 5)      # 고정하지 않습니다

t, tr = count(c)
print(f"전체 {t:,}개 중 학습 {tr:,}개 ({tr/t*100:.2f}%)")


## 4. 세 방식 비교표


In [ ]:
rows = []
for name, m in [("A 특징 추출만", a), ("B layer4 + fc", b), ("C 전체 미세조정", c)]:
    t, tr = count(m)
    rows.append({"방식": name, "전체 파라미터": f"{t:,}",
                 "학습 파라미터": f"{tr:,}", "비율": f"{tr/t*100:.2f}%"})
pd.DataFrame(rows)


## 5. 어느 층이 고정됐는지 확인하기


In [ ]:
rows = []
for name, child in b.named_children():
    params = list(child.parameters())
    if not params:
        continue
    frozen = all(not p.requires_grad for p in params)
    rows.append({"층": name, "상태": "고정" if frozen else "학습"})
pd.DataFrame(rows)


## 6. optimizer에는 학습할 것만 넘깁니다
고정한 파라미터를 넘겨도 오류는 안 나지만, 명시하는 편이 분명합니다.


In [ ]:
opt_all = torch.optim.Adam(a.parameters(), lr=1e-3)
opt_only = torch.optim.Adam(
    [p for p in a.parameters() if p.requires_grad], lr=1e-3
)
print("전체 넘김  파라미터 그룹 크기:", sum(p.numel() for g in opt_all.param_groups for p in g["params"]))
print("학습만 넘김 파라미터 그룹 크기:", sum(p.numel() for g in opt_only.param_groups for p in g["params"]))


## 7. 미세조정할 때는 학습률을 작게
이미 잘 학습된 가중치를 크게 흔들면 배운 것을 잃습니다.


In [ ]:
print("방식 A (fc만)      권장 학습률: 1e-3")
print("방식 B (일부)       권장 학습률: 1e-4")
print("방식 C (전체)       권장 학습률: 1e-4 ~ 1e-5")

# 층마다 다른 학습률을 줄 수도 있습니다
opt = torch.optim.Adam([
    {"params": c.layer4.parameters(), "lr": 1e-4},
    {"params": c.fc.parameters(),     "lr": 1e-3},
])
print("\n층별 학습률:", [g["lr"] for g in opt.param_groups])


## 8. 고정하면 학습이 빨라집니다
역전파를 계산할 층이 줄어듭니다.


In [ ]:
import time

x = torch.randn(16, 3, 224, 224)
loss_fn = nn.CrossEntropyLoss()
y = torch.randint(0, 5, (16,))

device = "cuda" if torch.cuda.is_available() else "cpu"
for name, m in [("A 특징 추출만", a), ("C 전체 미세조정", c)]:
    m = m.to(device)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-3)
    xb, yb = x.to(device), y.to(device)
    if device == "cuda": torch.cuda.synchronize()
    start = time.time()
    for _ in range(5):
        loss = loss_fn(m(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    if device == "cuda": torch.cuda.synchronize()
    print(f"{name:16s} 5회 {time.time()-start:.3f}초")


## 직접 해보기
1. `layer3` 과 `layer4` 를 함께 풀면 학습 파라미터가 몇 개가 되나요?
2. BatchNorm 층만 학습하도록 설정할 수 있나요? (이름에 "bn"이 들어간 파라미터만 풀기)


In [ ]:
# 여기에 작성하세요
